In [46]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [47]:
packages = [
    "io.delta:delta-spark_2.12:3.0.0",
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "com.amazonaws:aws-java-sdk-bundle:1.12.262"
]

In [48]:
spark = SparkSession.builder \
    .appName("silver_transformation") \
    .master("local[*]") \
    .config("spark.jars.packages", ",".join(packages)) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [49]:
def check_nulls(df):
    """
    Calculates the number of null values per columns.
    """
    df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])\
        .show()

def clean_text(col_name: str):
    """
    This function trims whitespace and converts to title case.
    """
    return initcap(trim(col(col_name)))

def clean_gender(col_name: str):
    """
    Normalizes gender values to specific categories.
    """
    c = upper(trim(col(col_name)))

    return when(c.isin("M", "MALE"), "Male") \
        .when(c.isin("F", "FEMALE"), "Female") \
        .otherwise("Unknown")

def handle_null_string(col_name: str, default_val: str = "Unknown"):
    """
    Fills NULL values with a default string.
    """
    return coalesce(clean_text(col_name), lit(default_val))

def clean_money(col_name: str):
    """
    Ensures monetary values are non-negative.
    """
    return abs(col(col_name))

def convert_validate_birthday(col_name: str):
    """
    Converts Debezium integer date offsets to actual dates and nulls out values outside the valid year range.
    """
    base = expr(f"date_add('1970-01-01', cast({col_name} as int))")
    curr_year = year(current_date())
    birth_year = year(base)

    return when((birth_year >= 1920) & (birth_year <= curr_year), base) \
        .otherwise(lit(None))

def clean_timestamp(col_name: str):
    """
    Convert Debezium timestamp to Spark timestamp.
    Postgres Debezium sends microseconds, Spark uses seconds/milliseconds.
    """
    return (col(col_name) / 1000000).cast("timestamp")

In [50]:
bronze_person_path = "s3a://bronze/person"
df_person = spark.read.format("delta").load(bronze_person_path)
df_person.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birthday: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- is_blocked: boolean (nullable = true)
 |-- create_time: long (nullable = true)
 |-- update_time: long (nullable = true)
 |-- cdc_operation: string (nullable = true)
 |-- cdc_timestamp: long (nullable = true)



In [51]:
df_person.select("id", "name", "gender", "birthday", "country", "city", "is_blocked").show(truncate=False)

+---+---------------------+-------+--------+-----------------+----------------------+----------+
|id |name                 |gender |birthday|country          |city                  |is_blocked|
+---+---------------------+-------+--------+-----------------+----------------------+----------+
|1  |jonathan munoz       |Male   |-2092   |Andorra          |new kathryn           |false     |
|2  |carol keller         |NULL   |7852    |Pakistan         |EAST DAVIDSIDE        |false     |
|3  |michelle green       |F      |-8945   |Cuba             |  Port Chelsea        |false     |
|4  |  Lorraine Harrison  |Female |576     |Mauritania       |  Port Leslie         |true      |
|5  |james osborne        |Unknown|11664   |Kenya            |EAST KEITHBURY        |false     |
|6  |eric andrews         |female |13573   |Sweden           |kristyburgh           |false     |
|7  |RENEE WATSON         |f      |12181   |Jersey           |  Jonesland           |false     |
|8  |carrie harrington    |Mal

In [52]:
total_count = df_person.count()
distinct_count = df_person.distinct().count()
print(f"Total rows: {total_count}")
print(f"Distinct rows: {distinct_count}")
print(f"Duplicate rows: {total_count - distinct_count}")

Total rows: 1087
Distinct rows: 1087
Duplicate rows: 0


In [53]:
df_person.groupBy("gender").count().orderBy(col("count").desc()).show()

+-------+-----+
| gender|count|
+-------+-----+
|      M|  158|
|   NULL|  136|
|      f|  124|
| female|  111|
|      F|  108|
|  Male |   99|
|      m|   92|
|   Male|   91|
| Female|   87|
|Unknown|   81|
+-------+-----+



In [54]:
df_person_clean = df_person.withColumn("name", clean_text("name")) \
    .withColumn("gender", clean_gender("gender")) \
    .withColumn("birthday", convert_validate_birthday("birthday")) \
    .withColumn("country", handle_null_string("country")) \
    .withColumn("city", handle_null_string("city"))

In [55]:
df_person_clean.select("id", "name", "gender", "birthday", "country", "city", "is_blocked").show(truncate=False)

+---+-----------------+-------+----------+-----------------+------------------+----------+
|id |name             |gender |birthday  |country          |city              |is_blocked|
+---+-----------------+-------+----------+-----------------+------------------+----------+
|1  |Jonathan Munoz   |Male   |1964-04-10|Andorra          |New Kathryn       |false     |
|2  |Carol Keller     |Unknown|1991-07-02|Pakistan         |East Davidside    |false     |
|3  |Michelle Green   |Female |1945-07-06|Cuba             |Port Chelsea      |false     |
|4  |Lorraine Harrison|Female |1971-07-31|Mauritania       |Port Leslie       |true      |
|5  |James Osborne    |Unknown|2001-12-08|Kenya            |East Keithbury    |false     |
|6  |Eric Andrews     |Female |2007-03-01|Sweden           |Kristyburgh       |false     |
|7  |Renee Watson     |Female |2003-05-09|Jersey           |Jonesland         |false     |
|8  |Carrie Harrington|Male   |1977-04-01|Somalia          |Lake Kaitlynberg  |true      |

In [56]:
check_nulls(df_person_clean)

+---+----+------+--------+-------+----+----------+-----------+-----------+-------------+-------------+
| id|name|gender|birthday|country|city|is_blocked|create_time|update_time|cdc_operation|cdc_timestamp|
+---+----+------+--------+-------+----+----------+-----------+-----------+-------------+-------------+
|  0|   0|     0|      14|      0|   0|         0|          0|          0|            0|            0|
+---+----+------+--------+-------+----+----------+-----------+-----------+-------------+-------------+



In [57]:
bronze_account_path = "s3a://bronze/account"
df_account = spark.read.format("delta").load(bronze_account_path)
df_account.printSchema()

root
 |-- id: integer (nullable = true)
 |-- owner_id: integer (nullable = true)
 |-- branch_id: integer (nullable = true)
 |-- account_type: string (nullable = true)
 |-- currency_code: string (nullable = true)
 |-- balance: double (nullable = true)
 |-- nickname: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- email: string (nullable = true)
 |-- create_time: long (nullable = true)
 |-- update_time: long (nullable = true)
 |-- is_blocked: boolean (nullable = true)
 |-- account_level: string (nullable = true)
 |-- cdc_operation: string (nullable = true)
 |-- cdc_timestamp: long (nullable = true)



In [58]:
df_account.select("id", "owner_id", "branch_id", "account_type", "currency_code", "balance", "nickname", "phone_number", "email").show(truncate=False)

+---+--------+---------+------------+-------------+---------------+--------+--------------------+---------------------------+
|id |owner_id|branch_id|account_type|currency_code|balance        |nickname|phone_number        |email                      |
+---+--------+---------+------------+-------------+---------------+--------+--------------------+---------------------------+
|209|91      |33       |V.I.P       |CAD          |8.746412702E7  |face    |(357) -843-2417     |carlos78@example.net       |
|9  |11      |21       |V.I.P       |VND          |2.871972671E7  |still   |964-262-9730x26484  |zmcclure@example.com       |
|3  |32      |15       |vip         |VND          |2.5059361708E8 |move    |898.859.1869x1021   |ocarney@example.com        |
|65 |81      |30       |sav         |EUR          |2.855374735E7  |similar |212-501-5521x345    |amercer@example.net        |
|158|146     |34       |vip         |VND          |1.61444027E8   |wind    |9024282069          |NULL                 

In [59]:
check_nulls(df_account)

+---+--------+---------+------------+-------------+-------+--------+------------+-----+-----------+-----------+----------+-------------+-------------+-------------+
| id|owner_id|branch_id|account_type|currency_code|balance|nickname|phone_number|email|create_time|update_time|is_blocked|account_level|cdc_operation|cdc_timestamp|
+---+--------+---------+------------+-------------+-------+--------+------------+-----+-----------+-----------+----------+-------------+-------------+-------------+
|  0|       0|        0|           0|            0|      0|       0|           0|  488|          0|          0|         0|            0|            0|            0|
+---+--------+---------+------------+-------------+-------+--------+------------+-----+-----------+-----------+----------+-------------+-------------+-------------+



In [60]:
df_account.groupBy("account_type").count().orderBy(col("count").desc()).show()

+------------+-----+
|account_type|count|
+------------+-----+
|         sav|  569|
|    Business|  569|
|       V.I.P|  550|
|         VIP|  513|
|      Saving|  477|
|         vip|  457|
|    checking|  441|
|    CHECKING|  419|
|   bussiness|  409|
|         Vip|  401|
|         chk|  399|
|    business|  363|
|      saving|  307|
+------------+-----+



In [61]:
df_account.groupBy("currency_code").count().orderBy(col("count").desc()).show()

+-------------+-----+
|currency_code|count|
+-------------+-----+
|          VND| 3732|
|          USD|  704|
|          EUR|  444|
|          JPY|  274|
|          CNY|  251|
|          AUD|  168|
|          SGD|   89|
|          CAD|   76|
|          GBP|   67|
|          THB|   40|
|          KRW|   29|
+-------------+-----+



In [62]:
total_count = df_account.count()
distinct_count = df_account.distinct().count()
print(f"Total rows: {total_count}")
print(f"Distinct rows: {distinct_count}")
print(f"Duplicate rows: {total_count - distinct_count}")

Total rows: 5874
Distinct rows: 5874
Duplicate rows: 0


In [63]:
def clean_phone(col_name: str):
    """
    Removes non-digit characters using Regex.
    """
    digits = regexp_replace(col(col_name), "[^0-9]", "")

    return when(
        (length(digits) >= 9) & (length(digits) <= 12),
        digits
    ).otherwise(lit(None))

def normalize_account_types(col_name: str):
    """ 
    Map messy types to standard categories.
    """
    c = upper(trim(col(col_name)))
    
    return when(c.rlike("CHECK|CHK"), "CHECKING") \
            .when(c.rlike("SAVING|SAV"), "SAVING") \
            .when(c.rlike("VIP|V\\.I\\.P"), "VIP") \
            .when(c.rlike("BUSINESS|BUSSINESS"), "BUSINESS") \
            .otherwise("UNKNOWN")
        

In [64]:
df_account_clean = df_account.withColumn("account_type", normalize_account_types("account_type")) \
    .withColumn("phone_number", clean_phone("phone_number")) \
    .withColumn("email", handle_null_string("email", "no-email@safebank.com")) \
    .withColumn("balance", clean_money("balance"))

In [65]:
df_account_clean.printSchema()

root
 |-- id: integer (nullable = true)
 |-- owner_id: integer (nullable = true)
 |-- branch_id: integer (nullable = true)
 |-- account_type: string (nullable = false)
 |-- currency_code: string (nullable = true)
 |-- balance: double (nullable = true)
 |-- nickname: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- email: string (nullable = false)
 |-- create_time: long (nullable = true)
 |-- update_time: long (nullable = true)
 |-- is_blocked: boolean (nullable = true)
 |-- account_level: string (nullable = true)
 |-- cdc_operation: string (nullable = true)
 |-- cdc_timestamp: long (nullable = true)



In [66]:
df_account_clean.select("id", "owner_id", "branch_id", "account_type", "account_level", "currency_code", "balance", "nickname", "phone_number", "email").show(truncate=False)

+---+--------+---------+------------+-------------+-------------+---------------+--------+------------+---------------------------+
|id |owner_id|branch_id|account_type|account_level|currency_code|balance        |nickname|phone_number|email                      |
+---+--------+---------+------------+-------------+-------------+---------------+--------+------------+---------------------------+
|209|91      |33       |VIP         |Stardard     |CAD          |8.746412702E7  |face    |3578432417  |Carlos78@example.net       |
|9  |11      |21       |VIP         |Silver       |VND          |2.871972671E7  |still   |NULL        |Zmcclure@example.com       |
|3  |32      |15       |VIP         |Gold         |VND          |2.5059361708E8 |move    |NULL        |Ocarney@example.com        |
|65 |81      |30       |SAVING      |Stardard     |EUR          |2.855374735E7  |similar |NULL        |Amercer@example.net        |
|158|146     |34       |VIP         |Gold         |VND          |1.61444027E

In [67]:
df_account_clean.groupBy("account_type").count().orderBy(col("count").desc()).show()

+------------+-----+
|account_type|count|
+------------+-----+
|         VIP| 1921|
|      SAVING| 1353|
|    BUSINESS| 1341|
|    CHECKING| 1259|
+------------+-----+



In [68]:
df_account_clean.groupBy("account_level").count().orderBy(col("count").desc()).show()

+-------------+-----+
|account_level|count|
+-------------+-----+
|         Gold| 2481|
|       Silver| 1860|
|     Stardard|  553|
|     Platinum|  534|
|     Standard|  428|
|          VIP|   18|
+-------------+-----+



In [69]:
check_nulls(df_account_clean)

+---+--------+---------+------------+-------------+-------+--------+------------+-----+-----------+-----------+----------+-------------+-------------+-------------+
| id|owner_id|branch_id|account_type|currency_code|balance|nickname|phone_number|email|create_time|update_time|is_blocked|account_level|cdc_operation|cdc_timestamp|
+---+--------+---------+------------+-------------+-------+--------+------------+-----+-----------+-----------+----------+-------------+-------------+-------------+
|  0|       0|        0|           0|            0|      0|       0|        3694|    0|          0|          0|         0|            0|            0|            0|
+---+--------+---------+------------+-------------+-------+--------+------------+-----+-----------+-----------+----------+-------------+-------------+-------------+



In [70]:
total_count = df_account_clean.count()
distinct_count = df_account_clean.distinct().count()

print(f"Total Rows: {total_count}")
print(f"Distinct Rows: {distinct_count}")
print(f"Duplicate Rows: {total_count - distinct_count}")

Total Rows: 5874
Distinct Rows: 5874
Duplicate Rows: 0


In [71]:
df_account_clean.groupBy("currency_code").agg(
    count("*").alias("count"),
    min("balance").alias("min_balance"),
    max("balance").alias("max_balance"),
    format_number(avg("balance"), 2).alias("avg_bal"),
    format_number(stddev("balance"), 2).alias("stddev_bal")
).orderBy("currency_code").show(truncate=False)

+-------------+-----+-----------+--------------+--------------+--------------+
|currency_code|count|min_balance|max_balance   |avg_bal       |stddev_bal    |
+-------------+-----+-----------+--------------+--------------+--------------+
|AUD          |168  |2559.63    |5.2210272663E8|101,686,663.65|130,746,120.11|
|CAD          |76   |1558.18    |3.8116067148E8|157,980,000.70|91,545,762.62 |
|CNY          |251  |148.48     |4.0317593874E8|100,592,723.52|107,232,225.63|
|EUR          |444  |594.89     |4.6857091431E8|97,039,972.96 |98,530,355.59 |
|GBP          |67   |553.19     |2.2944457141E8|84,240,803.79 |61,095,748.22 |
|JPY          |274  |11921.0    |3.8505972154E8|113,829,527.10|92,134,167.64 |
|KRW          |29   |799860.21  |2.0201868007E8|110,403,042.55|76,544,364.98 |
|SGD          |89   |195.11     |2.2863233434E8|82,126,893.31 |71,058,157.67 |
|THB          |40   |38021.17   |1.804347E8    |64,689,698.70 |57,248,750.22 |
|USD          |704  |0.16       |4.6595857652E8|87,9

In [72]:
df_account_clean.groupBy("currency_code").agg(
    expr("percentile_approx(balance, 0.25)").alias("25% (Q1)"),
    expr("percentile_approx(balance, 0.50)").alias("50% (Median)"),
    expr("percentile_approx(balance, 0.75)").alias("75% (Q3)"),
    expr("percentile_approx(balance, 0.95)").alias("Top 5% Rich"),
    expr("percentile_approx(balance, 0.99)").alias("Top 1% Whales")
).orderBy("currency_code").show(truncate=False)

+-------------+-------------+--------------+--------------+---------------+---------------+
|currency_code|25% (Q1)     |50% (Median)  |75% (Q3)      |Top 5% Rich    |Top 1% Whales  |
+-------------+-------------+--------------+--------------+---------------+---------------+
|AUD          |4594251.54   |4.832192057E7 |1.303933186E8 |4.355911072E8  |5.2210246211E8 |
|CAD          |8.746412702E7|1.7325780851E8|2.1732398658E8|3.4311694694E8 |3.8116067148E8 |
|CNY          |3259191.84   |7.103460656E7 |1.933352738E8 |2.9094043417E8 |4.031631843E8  |
|EUR          |1.381366723E7|7.520176773E7 |1.5537718087E8|2.9307263331E8 |4.4678388331E8 |
|GBP          |2.862355562E7|7.254013145E7 |1.5065910579E8|1.7071757674E8 |2.2944457141E8 |
|JPY          |2.61166585E7 |1.0423408429E8|1.7714944219E8|2.70551627E8   |3.8446998754E8 |
|KRW          |2.5938833E7  |1.1700557215E8|1.8232890315E8|1.9838868007E8 |2.0201868007E8 |
|SGD          |1.226053857E7|8.864355995E7 |9.691247612E7 |2.2862703351E8 |2.286

In [73]:
df_loan = spark.read.format("delta").load("s3a://bronze/loan_account")
df_loan.printSchema()

root
 |-- id: integer (nullable = true)
 |-- account_id: integer (nullable = true)
 |-- currency_code: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- interest_rate: double (nullable = true)
 |-- term_months: integer (nullable = true)
 |-- start_date: integer (nullable = true)
 |-- end_date: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- remaining_balance: double (nullable = true)
 |-- cdc_operation: string (nullable = true)
 |-- cdc_timestamp: long (nullable = true)



In [74]:
df_loan.count()

1229

In [75]:
df_loan.show(truncate=False)

+---+----------+-------------+-----------+-------------+-----------+----------+--------+------+-----------------+-------------+-------------+
|id |account_id|currency_code|amount     |interest_rate|term_months|start_date|end_date|status|remaining_balance|cdc_operation|cdc_timestamp|
+---+----------+-------------+-----------+-------------+-----------+----------+--------+------+-----------------+-------------+-------------+
|42 |132       |CHF          |30224.83   |11.49        |24         |20079     |20799   |ACTIVE|9732.95          |u            |1764226145284|
|47 |193       |CNY          |245269.67  |7.95         |12         |19808     |20168   |ACTIVE|60149.52         |u            |1764226152456|
|43 |73        |EUR          |80976.64   |7.7          |12         |20395     |20755   |ACTIVE|15457.92         |u            |1764226155011|
|32 |107       |GBP          |66212.63   |7.5          |12         |19694     |20054   |ACTIVE|23053.37         |u            |1764226159610|
|51 |1

In [76]:
print("Loan distribution")
df_loan.groupBy("status").count().orderBy(col("count").desc()).show()

Loan distribution
+-------+-----+
| status|count|
+-------+-----+
| ACTIVE| 1190|
|DEFAULT|   20|
| CLOSED|   19|
+-------+-----+



In [77]:
df_loan.filter(col("remaining_balance") > col("amount")).show()

+---+----------+-------------+------+-------------+-----------+----------+--------+------+-----------------+-------------+-------------+
| id|account_id|currency_code|amount|interest_rate|term_months|start_date|end_date|status|remaining_balance|cdc_operation|cdc_timestamp|
+---+----------+-------------+------+-------------+-----------+----------+--------+------+-----------------+-------------+-------------+
+---+----------+-------------+------+-------------+-----------+----------+--------+------+-----------------+-------------+-------------+



In [78]:
df_loan.filter((col("status") == "CLOSED") & (col("remaining_balance") > 0)).show()

+---+----------+-------------+------+-------------+-----------+----------+--------+------+-----------------+-------------+-------------+
| id|account_id|currency_code|amount|interest_rate|term_months|start_date|end_date|status|remaining_balance|cdc_operation|cdc_timestamp|
+---+----------+-------------+------+-------------+-----------+----------+--------+------+-----------------+-------------+-------------+
+---+----------+-------------+------+-------------+-----------+----------+--------+------+-----------------+-------------+-------------+



In [79]:
df_loan_clean = df_loan \
    .withColumn("amount", clean_money("amount")) \
    .withColumn("remaining_balance", clean_money("remaining_balance")) \
    .withColumn("start_date", convert_validate_birthday("start_date")) \
    .withColumn("end_date", convert_validate_birthday("end_date"))

df_loan_clean.show(truncate=False)     

+---+----------+-------------+-----------+-------------+-----------+----------+----------+------+-----------------+-------------+-------------+
|id |account_id|currency_code|amount     |interest_rate|term_months|start_date|end_date  |status|remaining_balance|cdc_operation|cdc_timestamp|
+---+----------+-------------+-----------+-------------+-----------+----------+----------+------+-----------------+-------------+-------------+
|42 |132       |CHF          |30224.83   |11.49        |24         |2024-12-22|NULL      |ACTIVE|9732.95          |u            |1764226145284|
|47 |193       |CNY          |245269.67  |7.95         |12         |2024-03-26|2025-03-21|ACTIVE|60149.52         |u            |1764226152456|
|43 |73        |EUR          |80976.64   |7.7          |12         |2025-11-03|NULL      |ACTIVE|15457.92         |u            |1764226155011|
|32 |107       |GBP          |66212.63   |7.5          |12         |2023-12-03|2024-11-27|ACTIVE|23053.37         |u            |1764226

In [80]:
invalid_dates = df_loan_clean.filter(col("start_date") > col("end_date"))
print(f"Loans with Start Date > End Date: {invalid_dates.count()}")
if invalid_dates.count() > 0:
    invalid_dates.show()

Loans with Start Date > End Date: 0


In [81]:
df_loan_clean.groupBy("currency_code").agg(
    sum("remaining_balance").alias("total_debt"),
    count("id").alias("total_loans")
).show()

+-------------+-------------------+-----------+
|currency_code|         total_debt|total_loans|
+-------------+-------------------+-----------+
|          GBP| 1437147.6099999996|        155|
|          CHF| 1496246.6300000001|        103|
|          CAD|         2206831.23|        107|
|          CNY|         8097214.13|         68|
|          EUR| 1470235.5699999998|        121|
|          THB|7.703235255000001E7|         95|
|          AUD|         1775736.17|         84|
|          KRW|      1.385458414E9|        106|
|          JPY|       2.73079274E8|        190|
|          USD|          261322.76|         58|
|          VND|    2.6888556208E10|         60|
|          SGD|  580753.7699999999|         82|
+-------------+-------------------+-----------+



In [82]:
tables = ["branch", "currency", "channel", "merchant", "device"]

for table in tables:
    df = spark.read.format("delta").load(f"s3a://bronze/{table}")
    print(f"Table {table}")
    print(f"Count: {df.count()}")
    df.printSchema()
    df.show(5, truncate=False)

Table branch
Count: 34
root
 |-- id: integer (nullable = true)
 |-- branch_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- open_date: integer (nullable = true)
 |-- cdc_operation: string (nullable = true)
 |-- cdc_timestamp: long (nullable = true)

+---+--------------------+-----------+---------+-------------+-------------+
|id |branch_name         |city       |open_date|cdc_operation|cdc_timestamp|
+---+--------------------+-----------+---------+-------------+-------------+
|1  |SafeBank Hai Phong  |Hai Phong  |19117    |c            |1764223570643|
|2  |SafeBank Ho Chi Minh|Ho Chi Minh|17065    |c            |1764223570646|
|3  |SafeBank Ca Mau     |Ca Mau     |18782    |c            |1764223570646|
|4  |SafeBank Gia Lai    |Gia Lai    |19566    |c            |1764223570646|
|5  |SafeBank Dong Thap  |Dong Thap  |18602    |c            |1764223570646|
+---+--------------------+-----------+---------+-------------+-------------+
only showing top 5 rows

Table cur

In [83]:
bronze_transfer_path = "s3a://bronze/transfer"
df_transfer = spark.read.format("delta").load(bronze_transfer_path)
df_transfer.printSchema()

root
 |-- id: integer (nullable = true)
 |-- txn_time: long (nullable = true)
 |-- create_time: long (nullable = true)
 |-- from_account_id: integer (nullable = true)
 |-- to_account_id: integer (nullable = true)
 |-- merchant_id: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency_code: string (nullable = true)
 |-- channel_id: integer (nullable = true)
 |-- comment: string (nullable = true)
 |-- status: string (nullable = true)
 |-- cdc_operation: string (nullable = true)
 |-- cdc_timestamp: long (nullable = true)



In [84]:
df_transfer.show(truncate=False)

+----+----------------+----------------+---------------+-------------+-----------+-----------+-------------+----------+--------------------------+-------+-------------+-------------+
|id  |txn_time        |create_time     |from_account_id|to_account_id|merchant_id|amount     |currency_code|channel_id|comment                   |status |cdc_operation|cdc_timestamp|
+----+----------------+----------------+---------------+-------------+-----------+-----------+-------------+----------+--------------------------+-------+-------------+-------------+
|1858|1764254676208733|1764229476204917|134            |79           |NULL       |295161.0   |JPY          |4         |Provide challenge stuff.  |SUCCESS|c            |1764229476337|
|1859|1764254679959973|1764229479957007|159            |NULL         |18         |2.1467655E7|VND          |1         |Payment Inv #1353         |SUCCESS|c            |1764229480425|
|1860|1764254682185710|1764229482182663|96             |350          |NULL       |9.9

In [85]:
df_transfer_clean = df_transfer.withColumn("amount", clean_money("amount"))

In [86]:
df_transfer_clean.head()

Row(id=1858, txn_time=1764254676208733, create_time=1764229476204917, from_account_id=134, to_account_id=79, merchant_id=None, amount=295161.0, currency_code='JPY', channel_id=4, comment='Provide challenge stuff.', status='SUCCESS', cdc_operation='c', cdc_timestamp=1764229476337)

In [88]:
df_transfer_clean = df_transfer_clean \
    .withColumn("txn_time", clean_timestamp("txn_time")) \
    .withColumn("create_time", clean_timestamp("create_time"))
df_transfer_clean.show(truncate=False)

+----+--------------------------+--------------------------+---------------+-------------+-----------+-----------+-------------+----------+--------------------------+-------+-------------+-------------+
|id  |txn_time                  |create_time               |from_account_id|to_account_id|merchant_id|amount     |currency_code|channel_id|comment                   |status |cdc_operation|cdc_timestamp|
+----+--------------------------+--------------------------+---------------+-------------+-----------+-----------+-------------+----------+--------------------------+-------+-------------+-------------+
|1858|2025-11-27 21:44:36.208733|2025-11-27 14:44:36.204917|134            |79           |NULL       |295161.0   |JPY          |4         |Provide challenge stuff.  |SUCCESS|c            |1764229476337|
|1859|2025-11-27 21:44:39.959973|2025-11-27 14:44:39.957007|159            |NULL         |18         |2.1467655E7|VND          |1         |Payment Inv #1353         |SUCCESS|c            |

In [89]:
fact_tables = ['sign_in', 'loan_payment', 'exchange_rate']

for table in fact_tables:
    df = spark.read.format("delta").load(f"s3a://bronze/{table}")
    print(f"Table {table}")
    print(f"Count: {df.count()}")
    df.printSchema()
    df.show(5, truncate=False)

Table sign_in
Count: 4532
root
 |-- id: integer (nullable = true)
 |-- account_id: integer (nullable = true)
 |-- device_id: integer (nullable = true)
 |-- sign_in_time: long (nullable = true)
 |-- ip_address: string (nullable = true)
 |-- location_city: string (nullable = true)
 |-- status: string (nullable = true)
 |-- cdc_operation: string (nullable = true)
 |-- cdc_timestamp: long (nullable = true)

+----+----------+---------+----------------+--------------+--------------+-------+-------------+-------------+
|id  |account_id|device_id|sign_in_time    |ip_address    |location_city |status |cdc_operation|cdc_timestamp|
+----+----------+---------+----------------+--------------+--------------+-------+-------------+-------------+
|1437|33        |50       |1764227436954208|186.201.149.34|Port Tristan  |SUCCESS|c            |1764227437015|
|1438|255       |11       |1764227440396735|151.99.58.215 |Williamland   |SUCCESS|c            |1764227440594|
|1439|56        |7        |17642274412